## Chat with an AI Model via Microsoft Foundry

In this notebook, you will learn how to send a question to an AI model hosted on Microsoft Foundry and get a response back. We will walk through every step, from installing libraries to reading the AI's answer. No prior experience is needed!

### Step 1: Install the Python Packages We Need

Before we can talk to the AI, we need to install a few helper libraries. Think of these as tools that let Python communicate with Microsoft Foundry and the OpenAI service.

In [ ]:
# This command downloads and installs the four packages we need:
# - azure-ai-projects: lets us connect to a Microsoft Foundry project
# - openai: the official OpenAI library for sending prompts and receiving answers
# - python-dotenv: reads secret values (like passwords) from a .env file so we don't hard-code them
# - azure-identity: handles logging in to Azure securely without typing a password in our code
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

### Step 2: Load Our Configuration

We keep sensitive information like our project endpoint and model name in a special `.env` file. This step reads those values into Python so we can use them later. This is a best practice -- you should never paste secrets directly into your code.

In [ ]:
# 'os' lets us access environment variables stored on our computer
import os

# 'load_dotenv' reads key-value pairs from a .env file and makes them available as environment variables
from dotenv import load_dotenv

# 'DefaultAzureCredential' automatically figures out how to log in to Azure for us
from azure.identity import DefaultAzureCredential

# 'AIProjectClient' is the main class we use to connect to a Microsoft Foundry project
from azure.ai.projects import AIProjectClient

# Actually load the .env file now -- this must run before we try to read any values from it
load_dotenv("/Users/macbookpro/dev/python/github/python/foundry-nett/00-Setup/.env")

# Quick check — make sure nothing is missing
my_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID")
resource_group = os.getenv("AZURE_RESOURCE_GROUP")
account_name = os.getenv("FOUNDRY_ACCOUNT_NAME")

print("Endpoint:", my_endpoint)
print("Subscription:", subscription_id)
print("Resource group:", resource_group)
print("Account name:", account_name)

# Read the Foundry project endpoint URL from our .env file
# This is the web address of our Foundry project in the cloud
my_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")

# Read the name of the AI model we want to use (for example, "gpt-4o")
my_model = os.getenv("MODEL_DEPLOYMENT_NAME")

### Step 3: Connect to Our Foundry Project

Now we create a client object that represents our connection to the Foundry project. Think of this like dialing a phone number -- we are telling Python *where* to connect and *how* to prove our identity.

In [ ]:
# Create the Foundry project client
# - 'endpoint' tells it the URL of our project (loaded from the .env file above)
# - 'credential' uses DefaultAzureCredential so Azure handles authentication for us automatically
foundry_client = AIProjectClient(
    endpoint=my_endpoint,
    credential=DefaultAzureCredential()
)

### Step 4: Get an OpenAI-Compatible Chat Client

The Foundry project client we just created can give us a ready-to-use OpenAI client. This client knows how to send messages to the AI model and receive answers. We do not need to manage API keys ourselves -- Foundry takes care of that behind the scenes.

In [ ]:
# Ask our Foundry client to build an OpenAI chat client for us
# This one line handles all the connection details, so we can focus on asking questions
chat_client = foundry_client.get_openai_client()

### Step 5: Ask the AI a Question and Print the Answer

This is the exciting part! We will send a question to the AI model and then display whatever it writes back. We give the model two things:
1. **Instructions** -- a short description of how it should behave (its "personality").
2. **Input** -- the actual question we want answered.

In [ ]:
# Send our question to the AI model and store whatever it says in 'ai_response'
# - 'model' tells the service which deployed model to use
# - 'instructions' sets the AI's behavior (like giving it a role to play)
# - 'input' is the actual question we want the AI to answer
ai_response = chat_client.responses.create(
    model=my_model,
    instructions="You are a friendly and knowledgeable assistant who gives clear, concise answers.",
    input="What are three practical uses of AI in everyday life?"
)

# Print the text the AI generated
# 'output_text' contains the plain-text answer from the model
print(f"The AI replied:\n{ai_response.output_text}")